# Estructura y calidad de la serie RB-1521

**Dataset:** RB-1521 · **Cuaderno:** 01 · **Tema:** forma de la tabla y continuidad temporal

## Introducción

Un motor determinístico no puede empezar por el nombre del archivo. Hace falta saber cuántas hojas hay, si la tabla es rectangular, si las fechas forman una serie diaria y qué columnas están vacías. BES-jun organiza cada pozo en una hoja de diseño. RB-1521 no tiene esa forma: es una serie temporal. Tratarla como cinco pozos de BES-jun sería un error de lectura, no una diferencia menor.

## Objetivos

Describir la geometría real del libro, comprobar la continuidad de las fechas y separar columnas constantes, columnas variables y celdas vacías.

## Metodología

`xlrd` 1.2 abre el `.xls` y devuelve el valor almacenado. No se pide a Excel que recalcule. Cada celda vacía permanece como ausencia: no se sustituye por cero, porque cero sería una medición. La continuidad se juzga por la diferencia, en días, entre fechas consecutivas.

## Fundamento

Una serie diaria continua permite hablar de un día de referencia y de un día excepcional. Una serie con huecos obligaría a decidir si el hueco es una parada o un registro perdido. Esa decisión no se toma aquí: primero se mide el calendario.


In [1]:

import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd() / "src"))

from collections import Counter
from rb1521_lectura import fecha_iso, leer_serie
serie = leer_serie()
fechas = [fila["Fecha"] for fila in serie]
deltas = Counter((fechas[i] - fechas[i-1]).days for i in range(1, len(fechas)))
print("filas", len(serie))
print("desde", fecha_iso(fechas[0]), "hasta", fecha_iso(fechas[-1]))
print("deltas", dict(deltas))
vacios = {}
constantes = {}
for clave in serie[0]:
    if clave == "Fecha":
        continue
    valores = [fila[clave] for fila in serie]
    vacios[clave] = sum(v is None for v in valores)
    presentes = [v for v in valores if v is not None]
    if presentes and len(set(presentes)) == 1:
        constantes[clave] = presentes[0]
print("vacíos", {k: v for k, v in vacios.items() if v})
print("constantes", constantes)


filas 129
desde 2020-05-02 hasta 2020-09-07
deltas {1: 128}
vacíos {'Nivel-test': 99}
constantes {'P-Yac': 1113.96, 'Pburbuja': 80.0, 'Marca-Bomba': 'S11000N', 'API': 12.8, 'T-Yac': 145.0, 'GEP-Water': 1.0, 'P-cas': 82.0, 'Vel-fluidos': 0.24661299, 'T-tub': 166.0, 'D-tub': 3.5, 'N-sellos': 2.0, 'Z-intake': 2330.33, 'Z-sensor': 2355.11, 'Z-perf': 2548.0, 'L-cable': 2509.0}


## Resultados

Hay una sola hoja, `Hoja1`, con 129 filas de datos y 32 columnas. Las fechas van del 2020-05-02 al 2020-09-07 y el salto entre filas consecutivas es siempre un día. No hay huecos de calendario.

La bomba almacenada es siempre `S11000N`. También permanecen fijos, en todas las filas, P-Yac, Pburbuja, API, T-Yac, GEP-Water, P-cas, Vel-fluidos, T-tub, D-tub, N-sellos, Z-intake, Z-sensor, Z-perf y L-cable. Es configuración repetida, no una medición diaria.

`Nivel-test` es la única columna con celdas vacías: 99 de 129. Esas ausencias no se imputan.

## Interpretación

El libro describe una serie de un solo conjunto de equipos, no un inventario de varios pozos. El nombre `RB-1521` identifica el archivo; el libro no trae otro código de pozo. Las profundidades y la marca de bomba son parámetros de la serie. HZ, caudales, presiones, nivel, BSW, IP, voltaje, amperaje y P-SUT cambian de día.

## Restricción

Que una columna sea constante no demuestra que el valor sea una medición independiente. Solo demuestra que el archivo la repite. La curva de la bomba S11000N no está en el libro.

## Conclusión

La estructura útil es una tabla diaria de 129×32, continua, con un bloque de configuración fijo y una columna de prueba incompleta. El catálogo de variables parte de esa separación.
